# VQ-VAE Semantic message Network

> A semantic message encoder implemented as VQ-VAE, which encodes the message into a discrete latent space.

In [1]:
#| default_exp models.msg_net

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import torch
from torch import nn
from torch.nn import functional as F


In [2]:
#| export
from typing import List, Callable, Union, Any, TypeVar, Tuple

Tensor = TypeVar('torch.tensor')

In [3]:
#| export
from torch import nn
from abc import abstractmethod

class BaseVAE(nn.Module):
    
    def __init__(self) -> None:
        super(BaseVAE, self).__init__()

    def encode(self, input: Tensor) -> List[Tensor]:
        raise NotImplementedError

    def decode(self, input: Tensor) -> Any:
        raise NotImplementedError

    def sample(self, batch_size:int, current_device: int, **kwargs) -> Tensor:
        raise NotImplementedError

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        raise NotImplementedError

    @abstractmethod
    def forward(self, *inputs: Tensor) -> Tensor:
        pass

    @abstractmethod
    def loss_function(self, *inputs: Any, **kwargs) -> Tensor:
        pass

In [ ]:
# #| export
# class VectorQuantizer(nn.Module):
#     """
#     Reference:
#     [1] https://github.com/deepmind/sonnet/blob/v2/sonnet/src/nets/vqvae.py
#     """
#     def __init__(self,
#                  num_embeddings: int,
#                  embedding_dim: int,
#                  beta: float = 0.25):
#         super(VectorQuantizer, self).__init__()
#         self.K = num_embeddings
#         self.D = embedding_dim
#         self.beta = beta

#         self.embedding = nn.Embedding(self.K, self.D)
#         self.embedding.weight.data.uniform_(-1 / self.K, 1 / self.K)

#     def forward(self, latents: Tensor) -> Tensor:
#         latents = latents.permute(0, 2, 3, 1).contiguous()  # [B x D x H x W] -> [B x H x W x D]
#         latents_shape = latents.shape
#         flat_latents = latents.view(-1, self.D)  # [BHW x D]

#         # Compute L2 distance between latents and embedding weights
#         dist = torch.sum(flat_latents ** 2, dim=1, keepdim=True) + \
#                torch.sum(self.embedding.weight ** 2, dim=1) - \
#                2 * torch.matmul(flat_latents, self.embedding.weight.t())  # [BHW x K]

#         # Get the encoding that has the min distance
#         encoding_inds = torch.argmin(dist, dim=1).unsqueeze(1)  # [BHW, 1]

#         # Convert to one-hot encodings
#         device = latents.device
#         encoding_one_hot = torch.zeros(encoding_inds.size(0), self.K, device=device)
#         encoding_one_hot.scatter_(1, encoding_inds, 1)  # [BHW x K]

#         # Quantize the latents
#         quantized_latents = torch.matmul(encoding_one_hot, self.embedding.weight)  # [BHW, D]
#         quantized_latents = quantized_latents.view(latents_shape)  # [B x H x W x D]

#         # Compute the VQ Losses
#         commitment_loss = F.mse_loss(quantized_latents.detach(), latents)
#         embedding_loss = F.mse_loss(quantized_latents, latents.detach())

#         vq_loss = commitment_loss * self.beta + embedding_loss

#         # Add the residue back to the latents
#         quantized_latents = latents + (quantized_latents - latents).detach()

#         return quantized_latents.permute(0, 3, 1, 2).contiguous(), vq_loss  # [B x D x H x W]


In [ ]:
#| export
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, beta=0.25, decay=0.99, eps=1e-5):
        super().__init__()
        self.K = num_embeddings
        self.D = embedding_dim
        self.beta = beta
        self.decay = decay
        self.eps = eps

        embedding = torch.randn(num_embeddings, embedding_dim)
        self.register_buffer('embedding', embedding)
        self.register_buffer('cluster_size', torch.zeros(num_embeddings))
        self.register_buffer('ema_embed', embedding.clone())

    def forward(self, latents):
        latents = latents.permute(0, 2, 3, 1).contiguous()
        latents_shape = latents.shape
        flat_latents = latents.view(-1, self.D)

        dist = (torch.sum(flat_latents ** 2, dim=1, keepdim=True)
                + torch.sum(self.embedding ** 2, dim=1)
                - 2 * torch.matmul(flat_latents, self.embedding.t()))

        encoding_inds = torch.argmin(dist, dim=1).unsqueeze(1)
        encoding_one_hot = torch.zeros(encoding_inds.size(0), self.K, device=latents.device)
        encoding_one_hot.scatter_(1, encoding_inds, 1)

        avg_probs = encoding_one_hot.mean(0).detach()  # [K] — average usage of each code
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        quantized_latents = torch.matmul(encoding_one_hot, self.embedding)
        quantized_latents = quantized_latents.view(latents_shape)

        if self.training:
            # EMA update — happens outside the gradient graph
            self.cluster_size.mul_(self.decay).add_(
                encoding_one_hot.sum(0).detach(), alpha=1 - self.decay
            )
            dw = torch.matmul(encoding_one_hot.t(), flat_latents.detach())
            self.ema_embed.mul_(self.decay).add_(dw, alpha=1 - self.decay)

            # Laplace smoothing to avoid dead codes
            n = self.cluster_size.sum()
            smoothed = (self.cluster_size + self.eps) / (n + self.K * self.eps) * n
            # self.embedding = self.ema_embed / smoothed.unsqueeze(1)
            self.embedding.copy_(self.ema_embed / smoothed.unsqueeze(1))

        # Only commitment loss — no embedding_loss needed
        vq_loss = self.beta * F.mse_loss(quantized_latents.detach(), latents)

        quantized_latents = latents + (quantized_latents - latents).detach()
        return quantized_latents.permute(0, 3, 1, 2).contiguous(), vq_loss, perplexity
    

In [8]:
#| export
# class VectorQuantizer(nn.Module):
#     """
#     Reference:
#     [1] https://github.com/deepmind/sonnet/blob/v2/sonnet/src/nets/vqvae.py
#     """
#     def __init__(self,
#                  num_embeddings: int,
#                  embedding_dim: int,
#                  beta: float = 0.25):
#         super(VectorQuantizer, self).__init__()
#         self.K = num_embeddings
#         self.D = embedding_dim
#         self.beta = beta

#         self.embedding = nn.Embedding(self.K, self.D)
#         self.embedding.weight.data.uniform_(-1 / self.K, 1 / self.K)

#     def forward(self, latents: Tensor) -> Tensor:
#         latents = latents.permute(0, 2, 3, 1).contiguous()  # [B x D x H x W] -> [B x H x W x D]
#         latents_shape = latents.shape
#         flat_latents = latents.view(-1, self.D)  # [BHW x D]

#         # Compute L2 distance between latents and embedding weights
#         dist = torch.sum(flat_latents ** 2, dim=1, keepdim=True) + \
#                torch.sum(self.embedding.weight ** 2, dim=1) - \
#                2 * torch.matmul(flat_latents, self.embedding.weight.t())  # [BHW x K]

#         # Get the encoding that has the min distance
#         encoding_inds = torch.argmin(dist, dim=1).unsqueeze(1)  # [BHW, 1]

#         # Convert to one-hot encodings
#         device = latents.device
#         encoding_one_hot = torch.zeros(encoding_inds.size(0), self.K, device=device)
#         encoding_one_hot.scatter_(1, encoding_inds, 1)  # [BHW x K]

#         # Quantize the latents
#         quantized_latents = torch.matmul(encoding_one_hot, self.embedding.weight)  # [BHW, D]
#         quantized_latents = quantized_latents.view(latents_shape)  # [B x H x W x D]

#         # Compute the VQ Losses
#         commitment_loss = F.mse_loss(quantized_latents.detach(), latents)
#         embedding_loss = F.mse_loss(quantized_latents, latents.detach())

#         vq_loss = commitment_loss * self.beta + embedding_loss

#         # Add the residue back to the latents
#         quantized_latents = latents + (quantized_latents - latents).detach()

#         return quantized_latents.permute(0, 3, 1, 2).contiguous(), vq_loss  # [B x D x H x W]

class ResidualLayer(nn.Module):

    def __init__(self,
                 in_channels: int,
                 out_channels: int):
        super(ResidualLayer, self).__init__()
        self.resblock = nn.Sequential(nn.Conv2d(in_channels, out_channels,
                                                kernel_size=3, padding=1, bias=False),
                                      nn.ReLU(True),
                                      nn.Conv2d(out_channels, out_channels,
                                                kernel_size=1, bias=False))

    def forward(self, input: Tensor) -> Tensor:
        return input + self.resblock(input)


class VQVAE(BaseVAE):

    def __init__(self,
                 in_channels: int,
                 embedding_dim: int,
                 num_embeddings: int,
                 hidden_dims: List = None,
                 beta: float = 0.25,
                 img_size: int = 64,
                 **kwargs) -> None:
        super(VQVAE, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.img_size = img_size
        self.beta = beta

        modules = []
        if hidden_dims is None:
            hidden_dims = [128, 256]

        # Build Encoder
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, out_channels=h_dim,
                              kernel_size=4, stride=2, padding=1),
                    nn.LeakyReLU())
            )
            in_channels = h_dim

        modules.append(
            nn.Sequential(
                nn.Conv2d(in_channels, in_channels,
                          kernel_size=3, stride=1, padding=1),
                nn.LeakyReLU())
        )

        for _ in range(6):
            modules.append(ResidualLayer(in_channels, in_channels))
        modules.append(nn.LeakyReLU())

        modules.append(
            nn.Sequential(
                nn.Conv2d(in_channels, embedding_dim,
                          kernel_size=1, stride=1),
                nn.LeakyReLU())
        )

        self.encoder = nn.Sequential(*modules)

        self.vq_layer = VectorQuantizer(num_embeddings,
                                        embedding_dim,
                                        self.beta)

        # Build Decoder
        modules = []
        modules.append(
            nn.Sequential(
                nn.Conv2d(embedding_dim,
                          hidden_dims[-1],
                          kernel_size=3,
                          stride=1,
                          padding=1),
                nn.LeakyReLU())
        )

        for _ in range(6):
            modules.append(ResidualLayer(hidden_dims[-1], hidden_dims[-1]))

        modules.append(nn.LeakyReLU())

        hidden_dims.reverse()

        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dims[i],
                                       hidden_dims[i + 1],
                                       kernel_size=4,
                                       stride=2,
                                       padding=1),
                    nn.LeakyReLU())
            )

        modules.append(
            nn.Sequential(
                nn.ConvTranspose2d(hidden_dims[-1],
                                   out_channels=3,
                                   kernel_size=4,
                                   stride=2, padding=1),
                nn.Tanh()))

        self.decoder = nn.Sequential(*modules)

    def encode(self, input: Tensor) -> List[Tensor]:
        """
        Encodes the input by passing through the encoder network
        and returns the latent codes.
        :param input: (Tensor) Input tensor to encoder [N x C x H x W]
        :return: (Tensor) List of latent codes
        """
        result = self.encoder(input)
        return [result]

    def decode(self, z: Tensor) -> Tensor:
        """
        Maps the given latent codes
        onto the image space.
        :param z: (Tensor) [B x D x H x W]
        :return: (Tensor) [B x C x H x W]
        """

        result = self.decoder(z)
        return result

    def forward(self, input: Tensor, **kwargs) -> List[Tensor]:
        encoding = self.encode(input)[0]
        quantized_inputs, vq_loss, perplexity = self.vq_layer(encoding)
        
        return [self.decode(quantized_inputs), input, vq_loss, perplexity]

    def loss_function(self,
                      *args,
                      **kwargs) -> dict:
        """
        :param args:
        :param kwargs:
        :return:
        """
        recons = args[0]
        input = args[1]
        vq_loss = args[2]
        perplexity = args[3]
        recons_loss = F.l1_loss(recons, input)

        loss = recons_loss + vq_loss
        return {'loss': loss,
                'Reconstruction_Loss': recons_loss,
                'VQ_Loss': vq_loss,
                'Perplexity': perplexity}

    def sample(self,
               num_samples: int,
               current_device: Union[int, str], **kwargs) -> Tensor:
        raise Warning('VQVAE sampler is not implemented.')

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        """
        Given an input image x, returns the reconstructed image
        :param x: (Tensor) [B x C x H x W]
        :return: (Tensor) [B x C x H x W]
        """

        return self.forward(x)[0]
    

In [9]:
#| hide
from c3jepa_wm.models.msg_net import VQVAE

model = VQVAE(
    img_size= 64,
    in_channels=3,
    embedding_dim=64,
    num_embeddings=512,
    hidden_channels=128,
    num_residual_blocks=2,
    commitment_cost=0.25,
)

In [10]:
#| hide
model

VQVAE(
  (encoder): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
      (1): LeakyReLU(negative_slope=0.01)
    )
    (1): Sequential(
      (0): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
      (1): LeakyReLU(negative_slope=0.01)
    )
    (2): Sequential(
      (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): LeakyReLU(negative_slope=0.01)
    )
    (3): ResidualLayer(
      (resblock): Sequential(
        (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): ReLU(inplace=True)
        (2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
    )
    (4): ResidualLayer(
      (resblock): Sequential(
        (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): ReLU(inplace=True)
        (2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [12]:
#| hide
import torch
sample = torch.randn(3, 3, 64, 64)
encoding = model.encode(sample)[0]

In [13]:
#| hide
quantized_inputs, vq_loss = model.vq_layer(encoding)

In [14]:
#| hide
# 1. Pass through the encoder
encoding = model.encode(sample)[0]

# 2. Replicate the internal VQ distance matching math
# Shape of encoding: [B, D, H, W] -> for 64x64 input with 2 strided layers, H=16, W=16
flat_latents = encoding.permute(0, 2, 3, 1).contiguous().view(-1, model.embedding_dim)

# Calculate distances to embedding dictionary entries
dist = (
    torch.sum(flat_latents**2, dim=1, keepdim=True)
    + torch.sum(model.vq_layer.embedding.weight**2, dim=1)
    - 2 * torch.matmul(flat_latents, model.vq_layer.embedding.weight.t())
)

# 3. Get the absolute discrete indices
discrete_token_ids = torch.argmin(dist, dim=1)

# 4. Reshape to see your discrete grid map!
# For a 64x64 input, your encoder downsamples by 4 (stride=2 twice), resulting in a 16x16 discrete token map


In [15]:
discrete_grid = discrete_token_ids.view(3, 16, 16)

print("Your discrete token map (integers from 0 to 511):")
print(discrete_grid.shape)

Your discrete token map (integers from 0 to 511):
torch.Size([3, 16, 16])


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()